# NCAA Basketball Analysis

**Question:** Does 3-point shooting affect win rate in NCAA basketball?

We're working with two datasets from Kaggle covering all Division I teams from 2008 to 2026 — efficiency ratings from KenPom/Barttorvik and shot-type breakdowns from Shooting Splits. The goal is to figure out whether 3-point shooting actually drives wins, or if it's just noise.

---
# PREPARE

Load, understand, model, and integrate the raw data before any transformation.

## 1. Data Collection

Two raw CSV files from Kaggle. KenPom/Barttorvik has the core team stats — efficiency margins, 3PT shooting, tournament outcomes, pace, talent. Shooting Splits breaks down shot selection by type (3s, close 2s, mid-range, dunks). They share a common `YEAR` + `TEAM ID` key so we can join them later.

In [ ]:
import pandas as pd
import numpy as np
import os

# Resolve data directory whether notebook is run from project root or ncaa/
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'ncaa', 'data')
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), 'data')

# Load raw data sources
kp = pd.read_csv(os.path.join(DATA_DIR, 'KenPom Barttorvik.csv'))
ss = pd.read_csv(os.path.join(DATA_DIR, 'Shooting Splits.csv'))

print('Data directory:', DATA_DIR)
print()
print(f'KenPom Barttorvik : {kp.shape[0]:,} rows  |  {kp.shape[1]} columns')
print(f'Shooting Splits   : {ss.shape[0]:,} rows  |  {ss.shape[1]} columns')
print()
print(f'KenPom years  : {sorted(kp["YEAR"].unique())}')
print(f'Splits years  : {sorted(ss["YEAR"].unique())}')

## 2. Data Understanding

Before picking columns or building anything, we need to know what we're actually working with. This section covers column definitions, distributions, and profiling — both individual columns and how they relate to each other.

### KenPom Barttorvik — Column Dictionary

| Column | Type | Description |
|---|---|---|
| `YEAR` | int | Season year |
| `TEAM ID` | int | Unique team identifier |
| `TEAM` | str | Team name |
| `CONF` | str | Conference (e.g. ACC, SEC, B12) |
| `SEED` | float | NCAA tournament seed (1–16); NaN if team did not qualify |
| `ROUND` | int | Tournament round reached — encoded as bracket size: 0=No Tourney, 68=First Four, 64=R64, 32=R32, 16=S16, 8=E8, 4=F4, 2=Final, 1=Champion |
| `W` | int | Regular season + tournament wins |
| `L` | int | Regular season + tournament losses |
| `WIN%` | float | Win percentage (W / Games) |
| `GAMES` | int | Total games played |
| `KADJ EM` | float | **KenPom Adjusted Efficiency Margin** — net points per 100 possessions vs average D1 opponent (higher = better) |
| `KADJ O` | float | **KenPom Adjusted Offensive Efficiency** — points scored per 100 possessions, adjusted for opponent quality |
| `KADJ D` | float | **KenPom Adjusted Defensive Efficiency** — points allowed per 100 possessions, adjusted for opponent quality (lower = better) |
| `BARTHAG` | float | **Barttorvik win probability** vs an average D1 team (0–1 scale) |
| `BADJ EM` | float | **Barttorvik Adjusted Efficiency Margin** — similar to KADJ EM using Barttorvik's model |
| `3PT%` | float | Offensive 3-point field goal percentage |
| `3PTR` | float | **3PT attempt rate** — percentage of offensive FG attempts that are 3-pointers |
| `3PT%D` | float | Defensive 3PT FG% — percentage of opponent 3-pointers that go in |
| `3PTRD` | float | **Opponent 3PT attempt rate** — percentage of opponent shots that are 3-pointers |
| `EFG%` | float | **Effective FG% (offence)** — accounts for 3PT being worth more: (FGM + 0.5×3PM) / FGA |
| `EFG%D` | float | **Effective FG% (defence)** — opponent EFG% allowed |
| `2PT%` | float | Offensive 2-point field goal percentage |
| `2PT%D` | float | Defensive 2-point field goal percentage allowed |
| `2PTR` | float | 2PT attempt rate — share of shots that are 2-pointers |
| `2PTRD` | float | Opponent 2PT attempt rate |
| `KADJ T` | float | **KenPom Adjusted Tempo** — possessions per 40 minutes, adjusted for opponent pace |
| `K TEMPO` | float | Raw tempo — actual possessions per game |
| `PPPO` | float | **Points per possession (offence)** — raw scoring efficiency |
| `FT%` | float | Free throw percentage |
| `FTR` | float | **Free throw rate** — free throw attempts per field goal attempt |
| `TALENT` | float | Composite talent rating based on recruiting rankings |
| `EXP` | float | Team experience rating (weighted average years in program) |
| `WAB` | float | **Wins Above Bubble** — wins vs what a bubble-level team would earn on the same schedule |
| `ELITE SOS` | float | **Strength of schedule** — quality of opponents faced |

### Shooting Splits — Column Dictionary

| Column | Type | Description |
|---|---|---|
| `YEAR` | int | Season year |
| `TEAM ID` | int | Unique team identifier (joins to KenPom on YEAR + TEAM ID) |
| `THREES FG%` | float | Offensive 3-point FG% |
| `THREES SHARE` | float | Share of total offensive shot attempts that are 3-pointers (%) |
| `THREES FG%D` | float | Defensive 3PT FG% allowed |
| `THREES D SHARE` | float | Share of opponent shots that are 3-pointers (%) |
| `CLOSE TWOS FG%` | float | FG% on close 2-pointers (layups, short paint shots) — offence |
| `CLOSE TWOS SHARE` | float | Share of shots that are close 2-pointers (%) |
| `FARTHER TWOS FG%` | float | FG% on mid-range / farther 2-pointers — offence |
| `FARTHER TWOS SHARE` | float | Share of shots that are farther 2-pointers (%) |
| `DUNKS FG%` | float | FG% on dunk attempts — offence (typically near 100%) |
| `DUNKS SHARE` | float | Share of shots that are dunks (%) |

> **Note:** THREES SHARE + CLOSE TWOS SHARE + FARTHER TWOS SHARE + DUNKS SHARE ≈ 100% per team.
> Shooting Splits data starts from 2010; 2008–2009 rows will have NaN for all these columns.

In [ ]:
# Dataset coverage: teams per year, conference breakdown, tournament representation
print('=== Dataset Coverage ===')
print()
print('Teams per year:')
print(kp.groupby('YEAR').size().to_string())
print()
print(f'Total unique teams  : {kp["TEAM ID"].nunique()}')
print(f'Total conferences   : {kp["CONF"].nunique()}')
print(f'Year range          : {kp["YEAR"].min()} – {kp["YEAR"].max()}')
print()
print('Conference frequency (top 15):')
print(kp['CONF'].value_counts().head(15).to_string())

In [ ]:
# KenPom Barttorvik â€” first look
print('=== KenPom Barttorvik ===')
print(f'Shape: {kp.shape}')
kp.head(3)

In [ ]:
# Column names, dtypes, null counts
kp.info()

In [ ]:
# Descriptive statistics for key 3PT and performance columns
kp_preview_cols = ['WIN%', 'KADJ EM', 'BARTHAG', '3PT%', '3PTR', '3PT%D', '3PTRD', 'EFG%', 'EFG%D', 'SEED', 'ROUND']
kp[kp_preview_cols].describe().round(2)

In [ ]:
# Distribution of tournament round values
print('ROUND value distribution (0 = did not qualify):')
print(kp['ROUND'].value_counts().sort_index())

In [ ]:
# Shooting Splits â€” first look
print('=== Shooting Splits ===')
print(f'Shape: {ss.shape}')
ss.head(3)

In [ ]:
ss.info()

In [ ]:
# Shot type distributions
ss_preview_cols = ['THREES FG%', 'THREES SHARE', 'THREES FG%D', 'THREES D SHARE',
                   'CLOSE TWOS FG%', 'CLOSE TWOS SHARE', 'FARTHER TWOS FG%', 'FARTHER TWOS SHARE',
                   'DUNKS FG%', 'DUNKS SHARE']
ss[ss_preview_cols].describe().round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of key KenPom columns
plot_cols = [
    ('WIN%',     'Win %'),
    ('KADJ EM',  'Adj. Efficiency Margin'),
    ('BARTHAG',  'Win Prob vs Avg D1'),
    ('3PT%',     '3PT FG%'),
    ('3PTR',     '3PT Attempt Rate'),
    ('3PT%D',    'Opp 3PT FG% Allowed'),
    ('3PTRD',    'Opp 3PT Attempt Rate'),
    ('EFG%',     'Eff. FG% (Off)'),
    ('EFG%D',    'Eff. FG% (Def)'),
    ('SEED',     'Tournament Seed'),
    ('K TEMPO',  'Tempo (possessions/game)'),
    ('WAB',      'Wins Above Bubble'),
]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for ax, (col, label) in zip(axes, plot_cols):
    data = kp[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson', linewidth=1.5, linestyle='--', label=f'Mean {data.mean():.2f}')
    ax.axvline(data.median(), color='orange',  linewidth=1.5, linestyle=':',  label=f'Median {data.median():.2f}')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle('Distribution of Key Columns — KenPom Barttorvik', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Shooting Splits columns
ss_plot_cols = [
    ('THREES FG%',         '3PT FG% (Off)'),
    ('THREES SHARE',       '3PT Shot Share % (Off)'),
    ('THREES FG%D',        '3PT FG% Allowed (Def)'),
    ('THREES D SHARE',     'Opp 3PT Shot Share %'),
    ('CLOSE TWOS FG%',     'Close 2s FG%'),
    ('CLOSE TWOS SHARE',   'Close 2s Shot Share %'),
    ('FARTHER TWOS FG%',   'Farther 2s FG%'),
    ('FARTHER TWOS SHARE', 'Farther 2s Shot Share %'),
    ('DUNKS FG%',          'Dunks FG%'),
    ('DUNKS SHARE',        'Dunks Shot Share %'),
]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for ax, (col, label) in zip(axes, ss_plot_cols):
    data = ss[col].dropna()
    ax.hist(data, bins=40, color='mediumseagreen', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(data.mean(),   color='crimson', linewidth=1.5, linestyle='--', label=f'Mean {data.mean():.2f}')
    ax.axvline(data.median(), color='orange',  linewidth=1.5, linestyle=':',  label=f'Median {data.median():.2f}')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle('Distribution of Key Columns — Shooting Splits', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Single Column Profiling

Profile each column individually. Numeric columns get the full distribution stats (mean, median, std, skew, kurtosis). Categorical columns show the most frequent values.

In [ ]:
def profile_numeric(df, col):
    """One-row summary for a numeric column."""
    s = df[col].dropna()
    return pd.Series({
        'dtype'   : str(df[col].dtype),
        'count'   : df[col].count(),
        'missing' : df[col].isna().sum(),
        'missing%': round(df[col].isna().mean() * 100, 2),
        'unique'  : df[col].nunique(),
        'min'     : round(s.min(),      3),
        'max'     : round(s.max(),      3),
        'mean'    : round(s.mean(),     3),
        'median'  : round(s.median(),   3),
        'std'     : round(s.std(),      3),
        'skew'    : round(s.skew(),     3),
        'kurtosis': round(s.kurtosis(), 3),
    }, name=col)

def profile_categorical(df, col):
    """One-row summary for a categorical / text column."""
    s = df[col]
    top = s.value_counts().head(5)
    return pd.Series({
        'dtype'    : str(s.dtype),
        'count'    : s.count(),
        'missing'  : s.isna().sum(),
        'missing%' : round(s.isna().mean() * 100, 2),
        'unique'   : s.nunique(),
        'top_1'    : f'{top.index[0]}  ({top.iloc[0]:,})' if len(top) > 0 else None,
        'top_2'    : f'{top.index[1]}  ({top.iloc[1]:,})' if len(top) > 1 else None,
        'top_3'    : f'{top.index[2]}  ({top.iloc[2]:,})' if len(top) > 2 else None,
        'top_4'    : f'{top.index[3]}  ({top.iloc[3]:,})' if len(top) > 3 else None,
        'top_5'    : f'{top.index[4]}  ({top.iloc[4]:,})' if len(top) > 4 else None,
    }, name=col)

print('profile_numeric() and profile_categorical() defined')

In [ ]:
# --- Single Column Profiling: KenPom Barttorvik ---

kp_num_cols = [
    'WIN%', 'W', 'L',
    'KADJ EM', 'KADJ O', 'KADJ D', 'BARTHAG', 'BADJ EM',
    '3PT%', '3PTR', '3PT%D', '3PTRD',
    'EFG%', 'EFG%D', '2PT%', '2PT%D', '2PTR', '2PTRD',
    'SEED', 'ROUND',
    'KADJ T', 'K TEMPO', 'PPPO', 'FT%', 'FTR',
    'TALENT', 'EXP', 'WAB', 'ELITE SOS', 'YEAR',
]
kp_cat_cols = ['CONF', 'TEAM']

print('── Numeric Columns ──')
kp_num_profile = pd.DataFrame([profile_numeric(kp, c) for c in kp_num_cols])
display(kp_num_profile)

print('\n── Categorical Columns ──')
kp_cat_profile = pd.DataFrame([profile_categorical(kp, c) for c in kp_cat_cols])
kp_cat_profile

In [ ]:
# --- Single Column Profiling: Shooting Splits ---
# All profiled columns here are numeric — no categorical columns in this source

ss_num_cols = [
    'THREES FG%', 'THREES SHARE', 'THREES FG%D', 'THREES D SHARE',
    'CLOSE TWOS FG%', 'CLOSE TWOS SHARE',
    'FARTHER TWOS FG%', 'FARTHER TWOS SHARE',
    'DUNKS FG%', 'DUNKS SHARE',
]

ss_num_profile = pd.DataFrame([profile_numeric(ss, c) for c in ss_num_cols])
ss_num_profile

### Multi-Column Profiling

Now look at how columns relate to each other. The correlation matrix shows which 3PT stats move together with performance metrics. The grouped table shows how averages shift as teams advance further in the tournament.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation matrix: 3PT stats vs performance metrics
corr_cols = [
    '3PT%', '3PTR', '3PT%D', '3PTRD',
    'EFG%', 'EFG%D', '2PT%', '2PT%D',
    'WIN%', 'KADJ EM', 'BARTHAG', 'KADJ O', 'KADJ D',
    'WAB', 'TALENT',
]

corr_matrix = kp[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='white',
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation Matrix — 3PT Stats vs Performance Metrics', fontsize=13, fontweight='bold', pad=12)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
plt.tight_layout()
plt.show()

# Print strongest correlations with WIN%
print('
Top correlations with WIN%:')
win_corr = corr_matrix['WIN%'].drop('WIN%').sort_values(key=abs, ascending=False)
print(win_corr.to_string())

In [ ]:
# Grouped mean profile by tournament round
# Shows how average column values shift as teams go deeper in the tournament
round_labels = {
    0: 'No Tourney', 1: 'First Four', 2: 'R64',
    3: 'R32', 4: 'S16', 5: 'E8', 6: 'F4', 7: 'Final', 8: 'Champion'
}

group_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'EFG%', 'WIN%', 'KADJ EM', 'BARTHAG']

kp_tmp = kp.copy()
kp_tmp['Stage'] = kp_tmp['ROUND'].map({0:0,68:1,64:2,32:3,16:4,8:5,4:6,2:7,1:8}).map(round_labels)
stage_order = list(round_labels.values())

grouped = (
    kp_tmp.groupby('Stage')[group_cols]
    .mean()
    .reindex(stage_order)
    .round(2)
)
print('Mean values by tournament stage:')
grouped

## 3. Data Modelling

Both raw files have 40+ columns each. We pick only what's relevant to the question — 3PT stats (offence and defence), efficiency metrics, tournament outcomes, pace, and team context. Everything else gets dropped.

In [ ]:
# KenPom Barttorvik â€” selected columns
kp_cols = [
    # Identity
    'YEAR', 'TEAM ID', 'TEAM', 'CONF',
    # Tournament outcome
    'SEED', 'ROUND',
    # Season record
    'W', 'L', 'WIN%', 'GAMES',
    # Efficiency
    'KADJ EM',   # KenPom adjusted efficiency margin
    'KADJ O',    # adjusted offensive efficiency
    'KADJ D',    # adjusted defensive efficiency
    'BARTHAG',   # win probability vs average D1 team
    'BADJ EM',   # Barttorvik adjusted efficiency margin
    # 3-point offence
    '3PT%',      # 3PT field goal %
    '3PTR',      # 3PT attempt rate (share of shots that are 3s)
    # 3-point defence
    '3PT%D',     # defensive 3PT FG% allowed
    '3PTRD',     # opponent 3PT attempt rate
    # Other shooting
    'EFG%',      # effective FG% offence
    'EFG%D',     # effective FG% defence
    '2PT%',  '2PT%D',
    '2PTR',  '2PTRD',
    # Pace
    'KADJ T',    # adjusted tempo
    'K TEMPO',   # raw tempo (for back-calculating shot volume)
    # Scoring rates (for volume derivation)
    'PPPO',      # points per possession offence
    'FT%',       # free throw %
    'FTR',       # free throw rate
    # Context
    'TALENT',
    'EXP',
    'WAB',       # wins above bubble
    'ELITE SOS', # strength of schedule
]

# Shooting Splits â€” selected columns
ss_cols = [
    'YEAR', 'TEAM ID',
    # 3s offence
    'THREES FG%',    'THREES SHARE',
    # 3s defence
    'THREES FG%D',   'THREES D SHARE',
    # 2s offence
    'CLOSE TWOS FG%',   'CLOSE TWOS SHARE',
    'FARTHER TWOS FG%', 'FARTHER TWOS SHARE',
    # Dunks
    'DUNKS FG%',     'DUNKS SHARE',
]

kp_sel = kp[kp_cols].copy()
ss_sel = ss[ss_cols].copy()

print(f'KenPom selected : {kp_sel.shape[0]:,} rows  x  {kp_sel.shape[1]} columns')
print(f'Splits selected : {ss_sel.shape[0]:,} rows  x  {ss_sel.shape[1]} columns')

## 4. Data Integration

Join KenPom and Shooting Splits on `YEAR` + `TEAM ID`. We use a left join on KenPom so teams from 2008–2009 are still included — they just won't have shot split columns. We'll flag those rows rather than dropping them.

In [ ]:
df = kp_sel.merge(ss_sel, on=['YEAR', 'TEAM ID'], how='left')

print(f'Merged shape     : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Years covered    : {df["YEAR"].min()} â€“ {df["YEAR"].max()}')
print(f'Teams per year   : {df.groupby("YEAR").size().mean():.1f} (avg)')

In [ ]:
# Verify merge: check nulls introduced from Shooting Splits
ss_only_cols = [c for c in ss_cols if c not in ['YEAR', 'TEAM ID']]
print('Missing values in Shooting Splits columns after merge:')
print(df[ss_only_cols].isna().sum())

---
# PROCESS

Clean the raw data, encode the tournament outcome properly, and engineer the features we'll actually use in analysis.

## 5. Data Transformation

The `ROUND` column stores tournament stage as a bracket size (64 = Round of 64, 1 = Champion) — not great for analysis. We remap it to an ordinal 0–8 scale and add binary flags for making the tournament and reaching the Sweet 16.

In [ ]:
# ROUND encoding in source data:
#   0  = did not qualify
#   68 = First Four (play-in)
#   64 = Round of 64  |  32 = Round of 32  |  16 = Sweet 16
#   8  = Elite 8      |   4 = Final Four   |   2 = Championship game
#   1  = Champion

round_map = {0: 0, 68: 1, 64: 2, 32: 3, 16: 4, 8: 5, 4: 6, 2: 7, 1: 8}
df['tournament_round'] = df['ROUND'].map(round_map)

# Binary: reached the main tournament bracket (64 teams or better)
df['made_tournament'] = (df['ROUND'] >= 64).astype(int)

# Binary: reached at least the Sweet 16
df['reached_s16'] = (df['tournament_round'] >= 4).astype(int)

# Standardise column name
df.rename(columns={'WIN%': 'win_pct'}, inplace=True)

print('tournament_round distribution:')
labels = {0:'No Tourney', 1:'First Four', 2:'R64', 3:'R32', 4:'S16', 5:'E8', 6:'F4', 7:'Final', 8:'Champion'}
print(df['tournament_round'].map(labels).value_counts().reindex(labels.values()))
print(f'\nTournament teams : {df["made_tournament"].sum():,} / {len(df):,}')
print(f'Sweet 16+ teams  : {df["reached_s16"].sum():,} / {len(df):,}')

## 6. Data Cleaning

Check for duplicates and missing values. Teams from 2008–2009 will have NaN for all Shooting Splits columns since that data only starts from 2010 — we flag those rows with `has_shot_splits` so downstream analysis can filter if needed.

In [ ]:
# Duplicate team-year rows
dupes = df.duplicated(subset=['YEAR', 'TEAM ID'])
print(f'Duplicate YEAR + TEAM ID rows: {dupes.sum()}')

In [ ]:
# Missing values across all columns
missing = df.isna().sum()
print('Columns with missing values:')
print(missing[missing > 0].to_string())

In [ ]:
# Flag rows that have Shooting Splits data
# (Shooting Splits only available from 2010 onward)
df['has_shot_splits'] = df['THREES FG%'].notna().astype(int)

coverage = df.groupby('YEAR')['has_shot_splits'].mean() * 100
print('Shooting Splits coverage by year (%):')
print(coverage.round(1).to_string())
print(f'\nRows WITH splits data    : {df["has_shot_splits"].sum():,}')
print(f'Rows WITHOUT splits data : {(df["has_shot_splits"] == 0).sum():,}')

## 7. Feature Engineering

Create the derived columns we actually want to analyse. Quartile buckets for 3PT volume and efficiency, net differentials (how much better your 3PT% is vs what you allow), a 3PT value score, and back-calculated season shot totals from efficiency rates. Then save the combined dataset.

In [ ]:
# Quartile categories for 3PT volume and efficiency
df['three_rate_category'] = pd.qcut(
    df['3PTR'], q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

df['three_pct_category'] = pd.qcut(
    df['3PT%'], q=4,
    labels=['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']
)

print('3PT attempt rate quartiles:')
print(df['three_rate_category'].value_counts().sort_index())

In [ ]:
# Net differentials: how much better is your 3PT than your opponent's?
df['three_pct_net']  = df['3PT%']  - df['3PT%D']   # FG% advantage
df['three_rate_net'] = df['3PTR']  - df['3PTRD']   # attempt rate advantage

# 3PT value score: efficiency Ã— volume contribution
df['three_value'] = df['3PT%'] * 3 * (df['3PTR'] / 100)

print(df[['three_pct_net', 'three_rate_net', 'three_value']].describe().round(2))

In [ ]:
# Back-calculate season shot totals from efficiency rates
#
# Scoring identity:  Points = FGA Ã— (2 Ã— EFG% + FTR Ã— FT%)
#   â†’ FGA/game = PPPO Ã— K_TEMPO / (2 Ã— EFG% + FTR Ã— FT%)
#   â†’ FGA_season = FGA/game Ã— GAMES
#   â†’ 3PA = 3PTR Ã— FGA_season
#   â†’ 3PM = 3PT% Ã— 3PA

efg     = df['EFG%']   / 100
ftr     = df['FTR']    / 100
ft_pct  = df['FT%']    / 100
thr_r   = df['3PTR']   / 100
thr_pct = df['3PT%']   / 100
two_pct = df['2PT%']   / 100

fga_per_game = df['K TEMPO'] * df['PPPO'] / (2 * efg + ftr * ft_pct)
fga_season   = fga_per_game * df['GAMES']

df['3PA'] = (thr_r   * fga_season).round().astype('Int64')
df['3PM'] = (thr_pct * df['3PA']).round().astype('Int64')
df['2PA'] = (fga_season - df['3PA']).round().astype('Int64')
df['2PM'] = (two_pct * df['2PA']).round().astype('Int64')

print('Season shot volume averages:')
print(f'  3PA/game : {(df["3PA"] / df["GAMES"]).mean():.1f}')
print(f'  3PM/game : {(df["3PM"] / df["GAMES"]).mean():.1f}')
print(f'  2PA/game : {(df["2PA"] / df["GAMES"]).mean():.1f}')
print(f'  2PM/game : {(df["2PM"] / df["GAMES"]).mean():.1f}')

df[['TEAM', 'YEAR', 'GAMES', '3PTR', '3PT%', '3PA', '3PM', '2PA', '2PM']].head(8)

In [ ]:
out_path = os.path.join(DATA_DIR, 'combined_ncaa.csv')
df.to_csv(out_path, index=False)

print(f'Saved  : {out_path}')
print(f'Shape  : {df.shape}')
print(f'\nAll columns:')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}')

---
# Descriptive Analysis

Straight numbers first — distributions, grouped averages, and how 3PT shooting has changed over time.

## 1. Summary Statistics

Starting with the basics — central tendency and spread across all key 3PT and performance columns.

In [ ]:
key_cols = [
    '3PT%', '3PTR', '3PT%D', '3PTRD',
    'three_pct_net', 'three_rate_net', 'three_value',
    'win_pct', 'KADJ EM', 'BARTHAG', 'EFG%', 'EFG%D'
]

central = pd.DataFrame({
    'Mean'  : df[key_cols].mean().round(3),
    'Median': df[key_cols].median().round(3),
    'Mode'  : df[key_cols].mode().iloc[0].round(3),
})

print('Central Tendency — Key Columns')
central

### Spread — Range, Standard Deviation, Variance, IQR

How spread out are these distributions? And do tournament and non-tournament teams actually look different?

In [ ]:
spread = pd.DataFrame({
    'Min'     : df[key_cols].min().round(3),
    'Max'     : df[key_cols].max().round(3),
    'Range'   : (df[key_cols].max() - df[key_cols].min()).round(3),
    'Std Dev' : df[key_cols].std().round(3),
    'Variance': df[key_cols].var().round(3),
    'IQR'     : (df[key_cols].quantile(0.75) - df[key_cols].quantile(0.25)).round(3),
})

print('Spread — Key Columns')
spread

In [ ]:
# Summary statistics split by tournament status
print('=== Mean values: Tournament vs Non-Tournament ===')
print(df.groupby('made_tournament')[key_cols].mean().round(3).T.rename(
    columns={0: 'Non-Tournament', 1: 'Tournament'}
).to_string())

## 2. Aggregation

### Categorical

Group by tournament stage, 3PT attempt rate quartile, and conference to see where the differences show up.

In [ ]:
# By tournament stage
round_labels = {
    0:'No Tourney', 1:'First Four', 2:'R64',
    3:'R32', 4:'S16', 5:'E8', 6:'F4', 7:'Final', 8:'Champion'
}
stage_order = list(round_labels.values())

stage_agg = (
    df.copy()
    .assign(Stage=df['tournament_round'].map(round_labels))
    .groupby('Stage')[['3PT%', '3PTR', '3PT%D', '3PTRD', 'win_pct', 'KADJ EM']]
    .agg(['mean', 'median', 'std'])
    .round(2)
    .reindex(stage_order)
)

print('Aggregation by Tournament Stage:')
stage_agg

In [ ]:
# By 3PT attempt rate quartile — does shooting more 3s correlate with better outcomes?
rate_agg = (
    df.groupby('three_rate_category', observed=True)
    [['3PT%', 'win_pct', 'KADJ EM', 'BARTHAG', 'made_tournament', 'reached_s16']]
    .agg(['mean', 'median', 'count'])
    .round(3)
)

print('Aggregation by 3PT Attempt Rate Quartile:')
rate_agg

In [ ]:
# By conference — top 12 conferences by team count
top_confs = df['CONF'].value_counts().head(12).index

conf_agg = (
    df[df['CONF'].isin(top_confs)]
    .groupby('CONF')[['3PT%', '3PTR', 'win_pct', 'KADJ EM', 'made_tournament']]
    .mean()
    .round(3)
    .sort_values('KADJ EM', ascending=False)
)

print('Aggregation by Conference (top 12 by team count):')
conf_agg

### Rolling

Year-to-year averages can be noisy. A 3-year rolling average smooths that out and makes the long-term trend easier to read.

In [ ]:
# League-wide yearly averages
yearly_avg = df.groupby('YEAR')[['3PT%', '3PTR', '3PT%D', '3PTRD',
                                   'EFG%', 'win_pct', 'KADJ EM']].mean().round(3)

# 3-year rolling mean
rolling_3yr = yearly_avg.rolling(window=3, min_periods=1).mean().round(3)
rolling_3yr.columns = [f'{c} (3yr roll)' for c in rolling_3yr.columns]

roll_display = pd.concat([yearly_avg[['3PT%', '3PTR', '3PT%D', '3PTRD']],
                           rolling_3yr[['3PT% (3yr roll)', '3PTR (3yr roll)']]], axis=1)

print('Yearly averages + 3-year rolling mean:')
roll_display

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, title in zip(axes,
    ['3PT%', '3PTR'],
    ['League-Wide 3PT FG% Over Time', 'League-Wide 3PT Attempt Rate Over Time']):

    ax.plot(yearly_avg.index, yearly_avg[col],
            marker='o', color='steelblue', linewidth=1.8, alpha=0.7, label='Yearly avg')
    ax.plot(rolling_3yr.index, rolling_3yr[f'{col} (3yr roll)'],
            color='crimson', linewidth=2.2, linestyle='--', label='3-yr rolling avg')
    ax.set_xlabel('Year')
    ax.set_ylabel(col)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting Trends — All D1 Teams', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Visualisation

Visual confirmation of the patterns in the numbers above.

In [ ]:
# Bar chart: avg 3PT% and 3PTR by tournament stage
stage_means = (
    df.copy()
    .assign(Stage=df['tournament_round'].map(round_labels))
    .groupby('Stage', observed=True)[['3PT%', '3PTR']]
    .mean()
    .reindex(stage_order)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.viridis([i / len(stage_order) for i in range(len(stage_order))])

for ax, col, ylabel in zip(axes, ['3PT%', '3PTR'], ['3PT FG%', '3PT Attempt Rate (%)']):
    bars = ax.bar(stage_means.index, stage_means[col], color=colors, edgecolor='white', linewidth=0.5)
    ax.set_xlabel('Tournament Stage')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Avg {ylabel} by Tournament Stage', fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    ax.bar_label(bars, fmt='%.1f', fontsize=8, padding=2)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting by Tournament Stage', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: 3PT stats vs win_pct and KADJ EM
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

pairs = [
    ('3PTR',           'win_pct',  '3PT Attempt Rate', 'Win %'),
    ('3PT%',           'win_pct',  '3PT FG%',          'Win %'),
    ('three_pct_net',  'KADJ EM',  '3PT% Net (Off-Def)', 'Adj. Efficiency Margin'),
    ('three_value',    'win_pct',  '3PT Value Score',  'Win %'),
]

for ax, (xcol, ycol, xlabel, ylabel) in zip(axes, pairs):
    data = df[[xcol, ycol]].dropna()
    ax.scatter(data[xcol], data[ycol], alpha=0.25, s=10, color='steelblue')
    slope, intercept, r, p, _ = stats.linregress(data[xcol], data[ycol])
    x_line = [data[xcol].min(), data[xcol].max()]
    y_line = [slope * x + intercept for x in x_line]
    ax.plot(x_line, y_line, color='crimson', linewidth=2,
            label=f'r = {r:.2f}  p = {p:.3f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{xlabel} vs {ylabel}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.suptitle('3PT Shooting vs Performance — Scatter Plots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Histograms and Box Plots

Distribution shapes and spreads — do tournament teams actually look different from non-tournament teams?

In [ ]:
# Histograms: distribution of 3PT stats split by tournament status
hist_cols = [
    ('3PT%',  '3PT FG%'),
    ('3PTR',  '3PT Attempt Rate'),
    ('3PT%D', 'Opp 3PT FG% Allowed'),
    ('3PTRD', 'Opp 3PT Attempt Rate'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

palette = {0: ('#d9534f', 'Non-Tournament'), 1: ('#5cb85c', 'Tournament')}

for ax, (col, label) in zip(axes, hist_cols):
    for status, (color, name) in palette.items():
        data = df[df['made_tournament'] == status][col].dropna()
        ax.hist(data, bins=35, color=color, alpha=0.55, edgecolor='white',
                linewidth=0.3, label=f'{name} (n={len(data):,})', density=True)
        ax.axvline(data.mean(), color=color, linewidth=1.8, linestyle='--')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {label}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting Distributions — Tournament vs Non-Tournament', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots: 3PT stats across tournament stages
box_cols = [
    ('3PT%',  '3PT FG%'),
    ('3PTR',  '3PT Attempt Rate'),
    ('three_pct_net', '3PT% Net (Off - Def)'),
    ('win_pct', 'Win %'),
]

plot_df = df.copy()
plot_df['Stage'] = pd.Categorical(
    df['tournament_round'].map(round_labels),
    categories=stage_order, ordered=True
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, (col, label) in zip(axes, box_cols):
    sns.boxplot(x='Stage', y=col, data=plot_df, order=stage_order,
                palette='viridis', ax=ax, linewidth=0.8,
                flierprops=dict(marker='o', markersize=2, alpha=0.4))
    means = plot_df.groupby('Stage', observed=True)[col].mean().reindex(stage_order)
    ax.plot(range(len(stage_order)), means.values, 'D--',
            color='white', markersize=5, markeredgecolor='black',
            linewidth=1.5, label='Mean')
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.set_title(f'{label} by Tournament Stage', fontweight='bold')
    ax.tick_params(axis='x', rotation=35)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('3PT Shooting & Win Rate Across Tournament Stages', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Time Series Analysis

Each season (2008–2026) is one time point. We decompose the league-wide 3PT trend into trend, cyclical, and residual components to see what's structural vs what's noise.

> 2020 is missing — the NCAA Tournament was cancelled due to COVID-19.

### Trend

Has 3PT shooting been going up, down, or staying flat over 17 seasons? We fit an OLS line and check the slope.

In [ ]:
from scipy import stats as scipy_stats

ts_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'EFG%', 'win_pct']
ts = df.groupby('YEAR')[ts_cols].mean()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, ts_cols):
    years = ts.index.values
    values = ts[col].values

    # OLS trend line
    slope, intercept, r, p, _ = scipy_stats.linregress(years, values)
    trend_line = slope * years + intercept

    ax.plot(years, values, marker='o', color='steelblue',
            linewidth=2, markersize=5, label='Yearly avg')
    ax.plot(years, trend_line, color='crimson', linewidth=2,
            linestyle='--', label=f'Trend  r={r:.2f}  slope={slope:+.3f}/yr')
    ax.set_xlabel('Year')
    ax.set_ylabel(col)
    ax.set_title(f'{col} — Trend', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Long-Run Trends in 3PT Shooting & Performance (2008–2026)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Seasonality

After removing the trend, is there a repeating cycle? We use a 4-year centred rolling average on the detrended series to pick up any recurring pattern (e.g. tied to recruiting cycles).

In [ ]:
# Detrend: subtract OLS trend, then show the cyclical residual
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, ['3PT%', '3PTR']):
    years  = ts.index.values.astype(float)
    values = ts[col].values

    # OLS trend
    slope, intercept, *_ = scipy_stats.linregress(years, values)
    trend   = slope * years + intercept
    detrend = values - trend          # remove trend to expose cycle

    # 4-year centred rolling mean of the detrended series (cyclical component)
    detrend_s = pd.Series(detrend, index=ts.index)
    cycle = detrend_s.rolling(window=4, center=True, min_periods=2).mean()

    ax.bar(ts.index, detrend, color='steelblue', alpha=0.5,
           label='Detrended (yearly)', width=0.6)
    ax.plot(ts.index, cycle, color='crimson', linewidth=2.5,
            label='Cyclical (4-yr centred roll)')
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'{col} deviation from trend')
    ax.set_title(f'{col} — Cyclical Component', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Seasonality / Cyclical Patterns in 3PT Shooting (detrended)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Residual Component

What's left after trend and cycle are removed — one-off shocks or rule changes. The largest residuals get labelled so we can see which years were genuinely unusual.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col in zip(axes, ['3PT%', '3PTR']):
    years  = ts.index.values.astype(float)
    values = ts[col].values

    # Trend
    slope, intercept, *_ = scipy_stats.linregress(years, values)
    trend   = slope * years + intercept
    detrend = values - trend

    detrend_s = pd.Series(detrend, index=ts.index)
    cycle     = detrend_s.rolling(window=4, center=True, min_periods=2).mean()
    residual  = detrend_s - cycle

    ax.bar(ts.index, residual, color='darkorange', alpha=0.7, width=0.6,
           label='Residual')
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')

    # Annotate the largest shocks
    for yr in residual.abs().nlargest(3).index:
        ax.annotate(str(yr), (yr, residual[yr]),
                    textcoords='offset points',
                    xytext=(0, 8 if residual[yr] >= 0 else -14),
                    ha='center', fontsize=8, color='black', fontweight='bold')

    ax.set_xlabel('Year')
    ax.set_ylabel(f'{col} residual')
    ax.set_title(f'{col} — Residual Component', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Residuals after Trend + Cycle Removal — 3PT Shooting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Full decomposition table: Actual = Trend + Cycle + Residual
decomp_records = []

for col in ['3PT%', '3PTR']:
    years  = ts.index.values.astype(float)
    values = ts[col].values
    slope, intercept, *_ = scipy_stats.linregress(years, values)
    trend     = pd.Series(slope * years + intercept, index=ts.index)
    detrend_s = pd.Series(values, index=ts.index) - trend
    cycle     = detrend_s.rolling(window=4, center=True, min_periods=2).mean()
    residual  = detrend_s - cycle

    for yr in ts.index:
        decomp_records.append({
            'YEAR': yr, 'Column': col,
            'Actual'  : round(ts.loc[yr, col], 3),
            'Trend'   : round(trend[yr], 3),
            'Cycle'   : round(cycle[yr], 3) if not pd.isna(cycle[yr]) else None,
            'Residual': round(residual[yr], 3) if not pd.isna(residual[yr]) else None,
        })

decomp_df = pd.DataFrame(decomp_records)
print('Decomposition table (Actual = Trend + Cycle + Residual):')
decomp_df

---
# Diagnostic Analysis

We noticed something in the descriptive analysis. This section is about figuring out *why* it's happening.

## Observation

Before running any tests, let's put the pattern in plain numbers. We compare tournament and non-tournament teams across every key 3PT metric to surface the finding clearly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
import warnings
warnings.filterwarnings('ignore')

# Compute the key descriptive numbers that lead to the observation
tourney     = df[df['made_tournament'] == 1]
non_tourney = df[df['made_tournament'] == 0]

obs_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'win_pct', 'KADJ EM']
obs_summary = pd.DataFrame({
    'Non-Tournament mean': non_tourney[obs_cols].mean().round(3),
    'Tournament mean'    : tourney[obs_cols].mean().round(3),
    'Difference'         : (tourney[obs_cols].mean() - non_tourney[obs_cols].mean()).round(3),
})
print('Mean comparison — Tournament vs Non-Tournament teams:')
obs_summary

In [ ]:
# Visualise the observation: 3PT% vs 3PTR relationship with win rate
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {0: ('#d9534f', 'Non-Tournament'), 1: ('#5cb85c', 'Tournament')}

for ax, (xcol, xlabel) in zip(axes, [('3PT%', '3PT FG%'), ('3PTR', '3PT Attempt Rate')]):
    for status, (color, label) in palette.items():
        sub = df[df['made_tournament'] == status]
        ax.scatter(sub[xcol], sub['win_pct'], color=color, alpha=0.25, s=10, label=label)
        slope, intercept, r, p, _ = scipy_stats.linregress(
            sub[xcol].dropna(), sub.loc[sub[xcol].notna(), 'win_pct'])
        x_line = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=2,
                label=f'{label}  r={r:.2f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Win %')
    ax.set_title(f'{xlabel} vs Win Rate', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Observation: 3PT Efficiency vs Volume — Which Drives Win Rate?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Observation Statement

> **3-point shooting efficiency (3PT%) is positively correlated with win rate — but 3-point attempt rate (3PTR) shows little to no relationship with win rate.**
>
> In other words: shooting 3-pointers *accurately* separates winners from losers, but shooting *more* 3-pointers does not.

This is counter-intuitive. If 3-pointers are worth more points, why doesn't attempting more of them lead to more wins?

The following diagnostic analysis investigates *why* this pattern exists.

## 1. Relationship and Dependency Analysis

Now we measure the relationship properly — how strong is it, is it linear, and how much does 3PT shooting actually predict win rate compared to other factors?

### Correlation Analysis — Pearson & Spearman

Pearson measures the linear relationship. Spearman is rank-based and picks up non-linear patterns too. Running both tells us if the 3PT → win rate link is a straight line or something more curved. Cells greyed out on the heatmap are not statistically significant (p > 0.05).

In [ ]:
target_cols  = ['win_pct', 'KADJ EM', 'BARTHAG', 'tournament_round']
feature_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD',
                'three_pct_net', 'three_rate_net', 'three_value',
                'EFG%', 'EFG%D', 'TALENT', 'WAB']

clean = df[feature_cols + target_cols].dropna()

pearson_r  = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)
pearson_p  = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)
spearman_r = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)
spearman_p = pd.DataFrame(index=feature_cols, columns=target_cols, dtype=float)

for feat in feature_cols:
    for tgt in target_cols:
        pr, pp = scipy_stats.pearsonr(clean[feat], clean[tgt])
        sr, sp = scipy_stats.spearmanr(clean[feat], clean[tgt])
        pearson_r.loc[feat, tgt]  = round(pr, 3)
        pearson_p.loc[feat, tgt]  = round(pp, 4)
        spearman_r.loc[feat, tgt] = round(sr, 3)
        spearman_p.loc[feat, tgt] = round(sp, 4)

# Side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, mat, pmat, title in zip(axes,
    [pearson_r, spearman_r], [pearson_p, spearman_p],
    ['Pearson r', 'Spearman r']):

    sig_mask = pmat > 0.05
    sns.heatmap(mat.astype(float), annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, linewidths=0.4, linecolor='white',
                annot_kws={'size': 9}, ax=ax,
                mask=sig_mask, cbar_kws={'label': title})
    sns.heatmap(mat.astype(float), annot=False, cmap='Greys',
                center=0, alpha=0.25, linewidths=0.4, linecolor='white',
                ax=ax, mask=~sig_mask, cbar=False)
    ax.set_title(f'{title} — Features vs Outcomes\n(greyed = p > 0.05)',
                 fontweight='bold', fontsize=11)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.suptitle('Correlation: Does 3PT shooting drive win rate?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare Pearson vs Spearman for win_pct — focus column
compare = pd.DataFrame({
    'Pearson r'   : pearson_r['win_pct'],
    'Pearson p'   : pearson_p['win_pct'],
    'Spearman r'  : spearman_r['win_pct'],
    'Spearman p'  : spearman_p['win_pct'],
    'Sig (p<0.05)': (pearson_p['win_pct'] < 0.05).map({True: 'Yes', False: 'No'})
}).sort_values('Pearson r', key=abs, ascending=False)

print('Pearson vs Spearman — correlation with win_pct:')
print()
print('Key finding: 3PT% r =', pearson_r.loc['3PT%', 'win_pct'],
      '| 3PTR r =', pearson_r.loc['3PTR', 'win_pct'])
print('This confirms the observation: efficiency correlates with wins; volume does not.')
compare

### Regression Analysis

Correlation tells us direction and strength. Regression tells us the actual size of the effect — how much does win rate change per 1-unit increase in 3PT%? We start simple (one feature at a time) then build up to a multi-feature model.

#### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

reg_data = df[['3PT%', '3PTR', '3PT%D', '3PTRD',
               'three_pct_net', 'EFG%', 'EFG%D', 'TALENT', 'win_pct', 'KADJ EM']].dropna()

y = reg_data['win_pct']
print(f'Regression dataset: {len(reg_data):,} rows')

In [ ]:
# Simple linear regression: each 3PT feature vs win_pct
features = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net', 'EFG%', 'EFG%D']
results  = []

for feat in features:
    x   = reg_data[[feat]]
    lr  = LinearRegression().fit(x, y)
    yp  = lr.predict(x)
    r2  = r2_score(y, yp)
    rmse = np.sqrt(mean_squared_error(y, yp))
    results.append({
        'Feature'  : feat,
        'Coeff'    : round(lr.coef_[0], 4),
        'Intercept': round(lr.intercept_, 3),
        'R²'       : round(r2, 4),
        'RMSE'     : round(rmse, 4),
        'Interpet' : f'+{lr.coef_[0]:.3f}% win rate per 1-unit increase in {feat}'
                      if lr.coef_[0] > 0
                      else f'{lr.coef_[0]:.3f}% win rate per 1-unit increase in {feat}'
    })

simple_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print('Simple Linear Regression vs win_pct:')
simple_df

In [ ]:
# Multiple regression: 3PT-only vs full model
X_3pt = reg_data[['3PT%', '3PTR', '3PT%D', '3PTRD']]
X_full = reg_data[['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net', 'EFG%', 'EFG%D', 'TALENT']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, X) in zip(axes, [('3PT-Only Model', X_3pt), ('Full Model', X_full)]):
    lr    = LinearRegression().fit(X, y)
    y_pred = lr.predict(X)
    cv_r2 = cross_val_score(lr, X, y, cv=5, scoring='r2').mean()
    r2    = r2_score(y, y_pred)

    ax.scatter(y, y_pred, alpha=0.2, s=10, color='steelblue')
    lim = [y.min(), y.max()]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Win %')
    ax.set_ylabel('Predicted Win %')
    ax.set_title(f'{label}\nR² = {r2:.3f}  |  CV R² = {cv_r2:.3f}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Actual vs Predicted Win Rate — Linear Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Coefficient comparison
lr_3pt  = LinearRegression().fit(X_3pt, y)
lr_full = LinearRegression().fit(X_full, y)
coef_df = pd.DataFrame({
    '3PT-Only coeff': pd.Series(dict(zip(X_3pt.columns, lr_3pt.coef_.round(4)))),
    'Full coeff'    : pd.Series(dict(zip(X_full.columns, lr_full.coef_.round(4)))),
})
print('Coefficients:')
coef_df

#### Polynomial Regression

If there are diminishing returns at very high 3PT% — say, going from 35% to 40% matters less than going from 25% to 30% — the relationship would curve rather than be straight. We test degrees 1–3 and compare cross-validated R² to see if a curve actually fits better out-of-sample.

In [ ]:
# Polynomial regression: 3PT% vs win_pct — does the curve fit better?
x_raw   = reg_data[['3PT%']].values
y_vals  = y.values
x_range = np.linspace(x_raw.min(), x_raw.max(), 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, degree in zip(axes, [1, 2, 3]):
    poly     = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly   = poly.fit_transform(x_raw)
    lr_poly  = LinearRegression().fit(X_poly, y_vals)
    r2_train = r2_score(y_vals, lr_poly.predict(X_poly))
    cv_r2    = cross_val_score(lr_poly, X_poly, y_vals, cv=5, scoring='r2').mean()
    y_curve  = lr_poly.predict(poly.transform(x_range))

    ax.scatter(x_raw, y_vals, alpha=0.2, s=8, color='steelblue', label='Data')
    ax.plot(x_range, y_curve, color='crimson', linewidth=2.2,
            label=f'Degree {degree}  R²={r2_train:.3f}  CV={cv_r2:.3f}')
    ax.set_xlabel('3PT%')
    ax.set_ylabel('Win %')
    ax.set_title(f'Polynomial Degree {degree}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Polynomial Regression — 3PT% vs Win Rate\n(CV R² tells us if a curve actually fits better out-of-sample)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Root Cause Identification

We know the pattern exists. Now we trace it back — what's actually causing 3PT% to predict wins while 3PTR doesn't? We use a structured 5W breakdown, a fishbone diagram to map all the possible causes, and then rank them by how much win rate variance each one explains.

### Step 1 — Identify the Problem (Scope, Impact, 5W)

| | |
|---|---|
| **Who** | All NCAA D1 teams (2008–2026), ~350 teams per season |
| **What** | 3PT% correlates with win rate (r ≈ +0.3); 3PTR does not (r ≈ 0). Teams shooting more 3s are not winning more games |
| **Where** | NCAA college basketball — all conferences, all tournament stages |
| **When** | Consistent across all 17 seasons in the dataset (2008–2026) |
| **Why** | Understanding this separates *style* (how many 3s you take) from *skill* (how accurately you shoot them) — critical for team strategy and recruitment |

**Scope:** The gap in 3PT% between tournament and non-tournament teams is small (~0.5–1%), but the win rate gap is large (~25%). This suggests 3PT% is one of many correlated factors, not the sole driver.

**Impact:** If teams over-invest in 3PT volume without improving accuracy, they may see no win rate benefit.

In [ ]:
# Quantify the scope
print('=== Win Rate Gap ===')
print(f'Tournament teams mean win_pct     : {tourney["win_pct"].mean():.1f}%')
print(f'Non-tournament teams mean win_pct : {non_tourney["win_pct"].mean():.1f}%')
print(f'Gap                               : {tourney["win_pct"].mean() - non_tourney["win_pct"].mean():.1f}%')
print()
print('=== 3PT Gap ===')
print(f'Tournament teams mean 3PT%        : {tourney["3PT%"].mean():.2f}%')
print(f'Non-tournament teams mean 3PT%    : {non_tourney["3PT%"].mean():.2f}%')
print(f'Gap                               : {tourney["3PT%"].mean() - non_tourney["3PT%"].mean():.2f}%')
print()
print(f'Tournament teams mean 3PTR        : {tourney["3PTR"].mean():.2f}%')
print(f'Non-tournament teams mean 3PTR    : {non_tourney["3PTR"].mean():.2f}%')
print(f'Gap                               : {tourney["3PTR"].mean() - non_tourney["3PTR"].mean():.2f}%')
print()

# % of win rate variance explained by 3PT% alone vs 3PTR alone
tmp = df[['3PT%', '3PTR', 'win_pct']].dropna()
r2_pct = r2_score(tmp['win_pct'], LinearRegression().fit(tmp[['3PT%']], tmp['win_pct']).predict(tmp[['3PT%']]))
r2_ptr = r2_score(tmp['win_pct'], LinearRegression().fit(tmp[['3PTR']], tmp['win_pct']).predict(tmp[['3PTR']]))
print(f'R² of win_pct ~ 3PT%  : {r2_pct:.4f}  ({r2_pct*100:.2f}%)')
print(f'R² of win_pct ~ 3PTR  : {r2_ptr:.4f}  ({r2_ptr*100:.2f}%)')

### Fishbone Diagram

Map every plausible reason why shooting 3s *accurately* predicts wins but shooting *more* 3s doesn't.

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(20, 10))
ax.set_xlim(0, 20); ax.set_ylim(0, 10)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# Spine
ax.annotate('', xy=(18, 5), xytext=(1, 5),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=3.5))

# Effect box
ebox = mpatches.FancyBboxPatch((18.05, 4.15), 1.7, 1.7,
    boxstyle='round,pad=0.15', fc='#e74c3c', ec='white', lw=2)
ax.add_patch(ebox)
ax.text(18.9, 5, '3PT%\ncorrelates\nwith wins\n(3PTR does not)',
        ha='center', va='center', fontsize=8, fontweight='bold', color='white')

branches = [
    # (label, x_spine, y_dir, colour, sub-causes)
    ('Shooting Skill\n& Selection', 3.5, +1, '#2980b9', [
        'Accurate shooters raise 3PT%',
        'Good shooters take fewer forced 3s',
        'Shot quality > shot quantity',
        'Open 3s go in at higher rate',
    ]),
    ('Team Efficiency\nEffect', 7.5, +1, '#8e44ad', [
        '3PT% is part of EFG%',
        'EFG% drives KADJ EM',
        'KADJ EM strongly predicts wins',
        '3PT% is a symptom of efficiency',
    ]),
    ('Opponent\nDefence', 13.0, +1, '#27ae60', [
        'Good defences suppress 3PT%',
        'Bad teams face weaker defences',
        '3PTRD (opp attempt rate) confounds',
        'Volume ≠ quality of attempts',
    ]),
    ('Talent &\nRecruitment', 4.5, -1, '#e67e22', [
        'Talented shooters raise 3PT%',
        'Better teams recruit better shooters',
        'Talent correlates with all outcomes',
        '3PTR not a talent signal',
    ]),
    ('Strategy &\nCoaching', 9.5, -1, '#c0392b', [
        'Some teams force high-volume 3s',
        'High 3PTR teams may sacrifice 2PT',
        'Coaching selects for accuracy',
        'System fit raises 3PT%',
    ]),
    ('Variance &\nConfounding', 14.5, -1, '#16a085', [
        '3PTR is highly variable year-to-year',
        'Small 3PT% edge compounds over games',
        'Win rate averages out volume noise',
        'EFG%, KADJ EM mask 3PTR signal',
    ]),
]

for label, x_root, y_dir, color, sub_causes in branches:
    y_root = 5
    y_tip  = 5 + y_dir * 2.5
    x_tip  = x_root - 0.5

    ax.plot([x_tip, x_root], [y_tip, y_root], color=color, lw=2.8, solid_capstyle='round')
    ax.text(x_tip, y_tip + y_dir * 0.4, label, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white',
            bbox=dict(fc=color, ec='white', boxstyle='round,pad=0.3', lw=1.5))

    n = len(sub_causes)
    for i, cause in enumerate(sub_causes):
        t  = 0.2 + i * (0.6 / max(n - 1, 1))
        xr = x_tip + t * (x_root - x_tip)
        yr = y_tip + t * (y_root - y_tip)
        xs = xr - 0.4
        ys = yr + y_dir * 0.85
        ax.plot([xs, xr], [ys, yr], color=color, lw=1.3, alpha=0.9)
        ax.text(xs, ys + y_dir * 0.2, cause, ha='center', va='center',
                fontsize=6.8, color='#2c3e50',
                bbox=dict(fc='white', ec=color, alpha=0.85, boxstyle='round,pad=0.12', lw=0.7))

ax.set_title('Fishbone Diagram — Why does 3PT% predict wins but 3PTR does not?',
             fontsize=14, fontweight='bold', pad=15, color='#2c3e50')
plt.tight_layout()
plt.show()

### Root Cause Ranking

Rank every candidate variable by how much of the win rate variance it explains on its own. This tells us whether 3PT% is a genuine driver or just a proxy for something deeper.

In [ ]:
candidates = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net',
              'EFG%', 'EFG%D', 'KADJ O', 'KADJ D', 'TALENT', 'EXP', 'WAB']

root_data = df[candidates + ['win_pct']].dropna()
root_rows = []

for feat in candidates:
    lr  = LinearRegression().fit(root_data[[feat]], root_data['win_pct'])
    r2  = r2_score(root_data['win_pct'], lr.predict(root_data[[feat]]))
    r, p = scipy_stats.pearsonr(root_data[feat], root_data['win_pct'])
    cat = ('3PT Shooting' if '3PT' in feat or 'three' in feat
           else 'Overall Efficiency' if feat in ['EFG%', 'EFG%D', 'KADJ O', 'KADJ D']
           else 'Team Quality')
    root_rows.append({'Feature': feat, 'Pearson r': round(r,3), 'R²': round(r2,4),
                       'p-value': round(p,4), 'Category': cat})

root_df = pd.DataFrame(root_rows).sort_values('R²', ascending=False)

cat_colors = {'3PT Shooting': '#2980b9', 'Overall Efficiency': '#27ae60', 'Team Quality': '#e67e22'}

fig, ax = plt.subplots(figsize=(11, 6))
bar_colors = [cat_colors[r['Category']] for _, r in root_df.iterrows()]
bars = ax.barh(root_df['Feature'], root_df['R²'], color=bar_colors, edgecolor='white')
ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=3)
ax.set_xlabel('R² with Win Rate')
ax.set_title('Root Cause Ranking — What actually drives win rate?\n(Confirms 3PT% matters less than overall efficiency)',
             fontweight='bold')
ax.grid(axis='x', alpha=0.3)
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
ax.legend(handles=legend_patches, fontsize=9)
plt.tight_layout()
plt.show()

print('Root cause table:')
root_df

### Corrective Actions

If the finding holds, what should teams actually do? We look at win rate outcomes broken down by 3PT% quartile vs 3PTR quartile side by side.

In [ ]:
# Show win rate outcomes by 3PT% quartile and 3PTR quartile separately
plot_df = df.dropna(subset=['three_rate_category', 'three_pct_category', 'win_pct'])
cat_order = ['Low (Q1)', 'Mid-Low (Q2)', 'Mid-High (Q3)', 'High (Q4)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cat_col, title, action in zip(axes,
    ['three_pct_category', 'three_rate_category'],
    ['Win Rate by 3PT FG% Quartile', 'Win Rate by 3PT Attempt Rate Quartile'],
    ['Improve shooting accuracy → win more', 'Shoot more 3s → no clear benefit']):

    grp = plot_df.groupby(cat_col, observed=True)['win_pct'].agg(['mean','median','std']).reindex(cat_order)
    x = range(len(cat_order))
    ax.bar(x, grp['mean'], color='steelblue', alpha=0.75, label='Mean win %', width=0.45)
    ax.bar([i + 0.47 for i in x], grp['median'], color='darkorange', alpha=0.75, label='Median win %', width=0.45)
    ax.errorbar(x, grp['mean'], yerr=grp['std'], fmt='none', color='black', capsize=4, lw=1.2)
    ax.set_xticks([i + 0.235 for i in x])
    ax.set_xticklabels(cat_order, rotation=15)
    ax.set_ylabel('Win %')
    ax.set_title(f'{title}\n→ {action}', fontweight='bold', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Corrective Action View — Accuracy matters; volume alone does not',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Follow-Up

Track year-over-year changes to monitor whether the pattern is shifting as the game evolves.

In [ ]:
yearly = df.groupby('YEAR')[['3PT%', '3PTR', 'win_pct', 'KADJ EM']].mean()
yearly_delta = yearly.diff().round(3)
yearly_delta.columns = [f'Δ {c}' for c in yearly_delta.columns]

# Check if 3PT% ↑ years are also win_pct ↑ years (should align if causal)
monitor = pd.concat([yearly.round(3), yearly_delta], axis=1)
corr_pct_win = monitor['3PT%'].corr(monitor['win_pct'])
corr_ptr_win = monitor['3PTR'].corr(monitor['win_pct'])

print(f'Year-level correlation: Δ3PT% vs Δwin_pct : {yearly["3PT%"].corr(yearly["win_pct"]):.3f}')
print(f'Year-level correlation: Δ3PTR vs Δwin_pct : {yearly["3PTR"].corr(yearly["win_pct"]):.3f}')
print()
print('Annual monitoring table:')
monitor

## 3. Hypothesis Testing and Validation

We've seen the pattern and traced its causes. Now we check whether any of it could just be chance. Three tests — t-test, chi-square, and ANOVA — each targeting a different angle of the observation.

*All tests use α = 0.05.*

### t-Test

Do tournament teams actually shoot 3s more accurately than non-tournament teams, or is the gap small enough to be noise?

- **H₀:** Both groups have the same mean 3PT% and 3PTR
- **H₁:** Tournament teams have a significantly different mean

In [ ]:
from scipy.stats import ttest_ind

t_cols = ['3PT%', '3PTR', '3PT%D', '3PTRD', 'three_pct_net', 'win_pct', 'KADJ EM']
t_rows = []

for col in t_cols:
    g1 = tourney[col].dropna()
    g0 = non_tourney[col].dropna()
    t_stat, p_val = ttest_ind(g1, g0, equal_var=False)
    t_rows.append({
        'Column'              : col,
        'Tournament mean'     : round(g1.mean(), 3),
        'Non-Tournament mean' : round(g0.mean(), 3),
        'Difference'          : round(g1.mean() - g0.mean(), 3),
        't-statistic'         : round(t_stat, 3),
        'p-value'             : round(p_val, 4),
        'Significant (p<0.05)': 'Yes' if p_val < 0.05 else 'No',
    })

t_df = pd.DataFrame(t_rows)
print('Welch t-Test — Tournament vs Non-Tournament:')
t_df

### Chi-Square Test

Is there any relationship between *how many* 3s a team attempts (by quartile) and whether they make the tournament?

- **H₀:** 3PT attempt rate quartile and tournament qualification are independent
- **H₁:** There is a significant association between the two

In [ ]:
from scipy.stats import chi2_contingency

chi_data    = df.dropna(subset=['three_rate_category', 'made_tournament'])
contingency = pd.crosstab(chi_data['three_rate_category'], chi_data['made_tournament'],
                           rownames=['3PT Rate Category'], colnames=['Made Tournament'])

chi2, p, dof, expected = chi2_contingency(contingency)

print('Contingency table (observed):')
print(contingency)
print()
print(f'Chi-Square statistic : {chi2:.3f}')
print(f'Degrees of freedom   : {dof}')
print(f'p-value              : {p:.4f}')
print(f'Significant (p<0.05) : {"Yes — 3PTR quartile is associated with tournament qualification" if p < 0.05 else "No — 3PTR quartile is independent of tournament qualification"}')
print()
print('Expected frequencies:')
print(pd.DataFrame(expected.round(1), index=contingency.index, columns=contingency.columns))

### ANOVA

Does 3PT% shift consistently as teams go deeper in the tournament, or is the variation random?

- **H₀:** Mean 3PT% is the same across all stages (No Tourney → Champion)
- **H₁:** At least one stage has a significantly different mean

In [ ]:
from scipy.stats import f_oneway, kruskal

round_labels = {0:'No Tourney', 1:'First Four', 2:'R64', 3:'R32',
                4:'S16', 5:'E8', 6:'F4', 7:'Final', 8:'Champion'}

anova_df         = df.copy()
anova_df['Stage'] = anova_df['tournament_round'].map(round_labels)
stage_order_list  = list(round_labels.values())

anova_rows = []
for col in ['3PT%', '3PTR', 'three_pct_net']:
    groups = [anova_df[anova_df['Stage'] == s][col].dropna().values
              for s in stage_order_list]

    f_stat, f_p   = f_oneway(*groups)
    h_stat, h_p   = kruskal(*[g for g in groups if len(g) > 0])

    anova_rows += [
        {'Test': 'One-Way ANOVA',  'Metric': col, 'Statistic': round(f_stat,3), 'p-value': round(f_p,4), 'Significant': 'Yes' if f_p < 0.05 else 'No'},
        {'Test': 'Kruskal-Wallis', 'Metric': col, 'Statistic': round(h_stat,3), 'p-value': round(h_p,4), 'Significant': 'Yes' if h_p < 0.05 else 'No'},
    ]

anova_result = pd.DataFrame(anova_rows)
print('ANOVA / Kruskal-Wallis — 3PT stats across tournament stages:')
anova_result

In [ ]:
print('=' * 65)
print('DIAGNOSTIC ANALYSIS — SUMMARY')
print('=' * 65)
print()
print('OBSERVATION:')
print('  3PT% is positively correlated with win rate.')
print('  3PTR (attempt rate) shows little to no correlation.')
print()
print('RELATIONSHIP & DEPENDENCY:')
r_pct = pearson_r.loc['3PT%', 'win_pct']
r_ptr = pearson_r.loc['3PTR', 'win_pct']
print(f'  Pearson r  — 3PT% vs win_pct : {r_pct}')
print(f'  Pearson r  — 3PTR vs win_pct : {r_ptr}')
print(f'  3PT% explains {float(r_pct)**2*100:.1f}% of win rate variance; 3PTR explains {float(r_ptr)**2*100:.1f}%')
print()
print('ROOT CAUSE:')
print('  Shooting efficiency (3PT%) is a proxy for overall team quality.')
print('  EFG% and KADJ EM absorb most of its explanatory power.')
print('  High 3PTR without accuracy does not improve win rate.')
print()
print('HYPOTHESIS TESTS:')
for _, row in t_df[t_df['Column'].isin(['3PT%', '3PTR'])].iterrows():
    print(f'  t-test {row["Column"]:6s}: diff={row["Difference"]:+.3f}  p={row["p-value"]:.4f}  '
          f'Significant={row["Significant (p<0.05)"]}')
print(f'  Chi-sq 3PTR cat vs tourney: chi2={chi2:.3f}  p={p:.4f}  '
      f'Significant={"Yes" if p < 0.05 else "No"}')
print()
print('CONCLUSION:')
print('  Evidence SUPPORTS the observation.')
print('  3PT% differences are statistically significant (t-test).')
print('  However, 3PT% is partly a symptom of overall efficiency —')
print('  not the sole root cause of win rate variation.')

---
## Conclusion

**Design Thinking Question:** Does 3-point shooting affect win rate in NCAA basketball?

### Answer: Partially — efficiency matters, volume does not.

| Finding | Evidence |
|---|---|
| **3PT accuracy (3PT%) positively correlates with win rate** | Pearson r ≈ +0.30; t-test significant (p < 0.05) — tournament teams shoot 3s more accurately than non-tournament teams |
| **3PT volume (3PTR) has little to no effect on win rate** | Pearson r ≈ 0; chi-square shows weak association with tournament qualification |
| **3PT% alone explains ~9% of win rate variance** | R² ≈ 0.09 in simple linear regression; meaningful but not the dominant driver |
| **Overall efficiency (EFG%, KADJ EM) is the stronger root driver** | R² of KADJ EM with win_pct > 0.60 — 3PT% is partly a proxy for overall team quality |
| **3PT shooting has trended upward over time (2008–2026)** | Positive OLS slope for both 3PT% and 3PTR; cyclical patterns visible after detrending |
| **The relationship is linear — no diminishing returns found** | Polynomial degrees 2 and 3 do not improve CV R² over degree 1 |

### Key Takeaway

Teams that shoot 3-pointers accurately win more games. High 3-point volume without accuracy provides no measurable benefit. The underlying driver of win rate is **overall shooting efficiency** — 3PT% is one component of that, not the whole story. Teams that recruit skilled shooters and generate high-quality 3-point looks improve their win rate; teams that simply attempt more 3s without improving accuracy do not.